<a href="https://colab.research.google.com/github/adishup/gen-ai-lab/blob/main/experiment%207.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate torch

In [ ]:
# ============================================================
# EXPERIMENT 7
# FINE-TUNING A PRETRAINED LANGUAGE MODEL
# ============================================================

import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


print("=" * 60)
print("EXPERIMENT 7 - FINE-TUNING LANGUAGE MODEL")
print("=" * 60)


# ============================================================
# 1. SELECT DEVICE
# ============================================================

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("\nGPU Available")
    print("GPU:", torch.cuda.get_device_name(0))

else:

    device = torch.device("cpu")

    print("\nUsing CPU")


# ============================================================
# 2. CREATE DOMAIN-SPECIFIC DATASET
# ============================================================

texts = [
    "Artificial intelligence is transforming modern technology.",
    "Machine learning is widely used in data science.",
    "Deep learning uses neural networks to learn patterns.",
    "Generative AI can create text and images.",
    "The football team won the championship.",
    "The cricket player scored a century.",
    "The tennis player won the tournament.",
    "The basketball team played an excellent match."
]

labels = [
    1, 1, 1, 1,
    0, 0, 0, 0
]


dataset = Dataset.from_dict({
    "text": texts,
    "label": labels
})


print("\nDataset created successfully.")
print("Number of samples:", len(dataset))


# ============================================================
# 3. LOAD BERT TOKENIZER
# ============================================================

print("\nLoading BERT tokenizer...")

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

print("Tokenizer loaded successfully.")


# ============================================================
# 4. TOKENIZE DATASET
# ============================================================

def tokenize_function(example):

    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


tokenized_dataset = dataset.map(
    tokenize_function
)


# ============================================================
# 5. LOAD BERT CLASSIFICATION MODEL
# ============================================================

print("\nLoading BERT classification model...")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


model = model.to(device)

print("BERT model loaded successfully.")


# ============================================================
# 6. DEFINE LABEL MEANINGS
# ============================================================

label_names = {
    0: "SPORTS",
    1: "TECHNOLOGY"
}


# ============================================================
# 7. TRAINING CONFIGURATION
# ============================================================

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    learning_rate=2e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)


# ============================================================
# 8. CREATE TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)


# ============================================================
# 9. FINE-TUNE MODEL
# ============================================================

print("\nStarting model fine-tuning...")

trainer.train()

print("\nFine-tuning completed successfully.")


# ============================================================
# 10. SAVE FINE-TUNED MODEL
# ============================================================

save_path = "./fine_tuned_model"

trainer.save_model(
    save_path
)

tokenizer.save_pretrained(
    save_path
)

print("\nFine-tuned model saved to:")
print(save_path)


# ============================================================
# 11. LOAD SAVED MODEL
# ============================================================

print("\nLoading fine-tuned model...")

fine_tuned_tokenizer = AutoTokenizer.from_pretrained(
    save_path
)

fine_tuned_model = AutoModelForSequenceClassification.from_pretrained(
    save_path
)

fine_tuned_model = fine_tuned_model.to(device)

fine_tuned_model.eval()

print("Fine-tuned model loaded successfully.")


# ============================================================
# 12. PREDICTION FUNCTION
# ============================================================

def predict(text):

    inputs = fine_tuned_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = fine_tuned_model(
            **inputs
        )

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[
        0,
        predicted_class
    ].item()

    return (
        label_names[predicted_class],
        confidence
    )


# ============================================================
# 13. TEST THE FINE-TUNED MODEL
# ============================================================

print("\n" + "=" * 60)
print("TESTING FINE-TUNED MODEL")
print("=" * 60)


test_text = input(
    "\nEnter a sentence for classification: "
)


predicted_class, confidence = predict(
    test_text
)


# ============================================================
# 14. DISPLAY RESULT
# ============================================================

print("\nInput:")
print(test_text)

print("\nPredicted Class:")
print(predicted_class)

print("\nConfidence Score:")
print(round(confidence, 4))


# ============================================================
# 15. COMPLETION
# ============================================================

print("\n" + "=" * 60)
print("EXPERIMENT 7 COMPLETED SUCCESSFULLY")
print("=" * 60)

EXPERIMENT 7 - FINE-TUNING LANGUAGE MODEL

Using CPU

Dataset created successfully.
Number of samples: 8

Loading BERT tokenizer...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


Fine-tuning completed successfully.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuned model saved to:
./fine_tuned_model

Loading fine-tuned model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Fine-tuned model loaded successfully.

TESTING FINE-TUNED MODEL

Enter a sentence for classification: Artificial intelligence is improving modern technology.

Input:
Artificial intelligence is improving modern technology.

Predicted Class:
TECHNOLOGY

Confidence Score:
0.5257

EXPERIMENT 7 COMPLETED SUCCESSFULLY
